In [32]:
import numpy as np
import random

In [33]:
# Step 1:
states = np.arange(16, 31)
actions = ['ON','OFF']

In [34]:
print (states)
print (actions)

[16 17 18 19 20 21 22 23 24 25 26 27 28 29 30]
['ON', 'OFF']


In [35]:
# Step 2: 
# Q table is [s,a] state, action
Q = np.zeros((len(states),len(actions)))

In [45]:
# et alpha, gamma, epsilon, episodes
alpha=0.1
gamma=0.9
epsilon=0.2
episodes=300

#### 💡 Meaning:

    alpha (α) → how fast we learn from new rewards.
    (small = slow learning, large = overwrite old info quickly)

    gamma (γ) → how much we care about future rewards.
    close to 1 = think long-term)

    epsilon (ε) → how often to explore random actions.
    (helps avoid getting stuck in one habit)

    episodes → how many times the agent will repeat training.

In [37]:
# Step 3: Write Reward Function
def get_reward(temp, action):
    if 20 <= temp <= 24:
        reward = 10  # comfortable
    else:
        reward = -5  # uncomfortable

    if action == 'ON':
        reward -= 2  # cost of energy

    return reward

In [38]:
# Step 4: Environment Dynamic Change (e.g., Temperature Changes)
def next_temp(temp, action):
    if action == 'ON':
        temp -= random.choice([1, 2])   # Cooling effect
    else:
        temp += random.choice([0, 1, 2])  # Temperature may rise
    
    # Keep temperature within realistic limits (16°C–30°C)
    return int(np.clip(temp, 16, 30))


In [39]:
# Step 5: Training Loop
for ep in range(episodes):  # Loop over all episodes (training rounds)
    temp = random.choice(states)  # Start from a random temperature
    done = False

    for _ in range(20):  # Limit steps per episode
        # Choose action (epsilon-greedy)
        if random.uniform(0, 1) < epsilon:
            # Exploration: choose a random action
            action = random.choice(actions)
        else:
            # Exploitation: choose best action from Q-table
            action = actions[np.argmax(Q[temp - 16])]

        # Environment changes temperature based on action
        next_state = next_temp(temp, action)

        # Get reward for the new state
        reward = get_reward(next_state, action)

        # --- Update the Q-table (learning step) ---

        # Find the position (number) of the action we took — for example:
        # 'OFF' might be 0, 'ON' might be 1
        a = actions.index(action)

        # Find the best (highest) Q-value for the next temperature
        # This means: what’s the best possible future reward if we act smart next time
        best_next = np.max(Q[next_state - 16])

        # Update what we learned in the Q-table:
        # New value = Old value + learning rate * (reward + future reward - old value)
        # alpha = how fast we learn
        # gamma = how much we care about the future reward
        Q[temp - 16, a] += alpha * (reward + gamma * best_next - Q[temp - 16, a])
        
        # Move to the next temperature (the environment changes)
        temp = next_state



print('Training Complete')

Training Complete


In [40]:
# Test what the model learned

temp = 28  # starting temperature

for step in range(10):
    # pick the best action (ON or OFF) based on what the model learned
    action = actions[np.argmax(Q[temp - 16])]
    
    # show what the model does at this step
    print(f"Step {step+1}: Temp={temp}C -> Action={action}")
    
    # update the temperature depending on the action
    temp = next_temp(temp, action)


Step 1: Temp=28C -> Action=ON
Step 2: Temp=27C -> Action=ON
Step 3: Temp=26C -> Action=ON
Step 4: Temp=24C -> Action=ON
Step 5: Temp=23C -> Action=ON
Step 6: Temp=21C -> Action=OFF
Step 7: Temp=22C -> Action=ON
Step 8: Temp=20C -> Action=OFF
Step 9: Temp=22C -> Action=ON
Step 10: Temp=21C -> Action=OFF


In [44]:
# Ask user for starting temperature
try:
    temp = int(input('Enter Starting Room Temp (16-30): '))
    if temp < 16 or temp > 30:
        raise ValueError('Temp out of range')
except ValueError as ve:
    print(ve)
    temp = 25
    print('Set to default 25°C')

print(f'\nStarting temp: {temp}°C\n')

# Simulate 10 steps using the learned Q-table
for step in range(10):
    # Choose the best action for current temperature
    action = actions[np.argmax(Q[temp - 16])]
    
    # Print the current step, temperature, and action
    print(f"Step {step+1}: Temp={temp}°C -> Action={action}")
    
    # Update temperature based on action
    temp = next_temp(temp, action)

print('\nDone')


Enter Starting Room Temp (16-30):  15


Temp out of range
Set to default 25°C

Starting temp: 25°C

Step 1: Temp=25°C -> Action=ON
Step 2: Temp=24°C -> Action=ON
Step 3: Temp=23°C -> Action=ON
Step 4: Temp=22°C -> Action=ON
Step 5: Temp=21°C -> Action=OFF
Step 6: Temp=21°C -> Action=OFF
Step 7: Temp=22°C -> Action=ON
Step 8: Temp=21°C -> Action=OFF
Step 9: Temp=21°C -> Action=OFF
Step 10: Temp=22°C -> Action=ON

Done
